In [20]:
import pandas as pd
import sqlite3
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Phase 0: Loading Data

In [21]:
confirmatory_df = pd.read_csv('jd64p-osfstorage-archive/upworthy-archive-datasets/upworthy-archive-confirmatory-packages-03.12.2020.csv', 
                              low_memory=False)

confirmatory_df = confirmatory_df.drop(columns=['Unnamed: 0'])

conn = sqlite3.connect('upworthy.db')

# Write the dataframe into a table called 'packages'
confirmatory_df.to_sql('packages', conn, if_exists='replace', index=False)

105551

In [22]:
cursor = conn.cursor()
cursor.execute('SELECT COUNT(*) FROM packages')
print(cursor.fetchone())  # -> (105551,)

cursor.execute('SELECT COUNT(DISTINCT clickability_test_id) FROM packages')
print(cursor.fetchone())  # -> (22743,)

conn.close()

(105551,)
(22743,)


# Phase 1: Defining Treatment/Control

In [23]:
confirmatory_df = confirmatory_df.reset_index(drop=True)
confirmatory_df['package_id'] = confirmatory_df.index + 1

# --- Feature engineering ---
analyzer = SentimentIntensityAnalyzer()

confirmatory_df['has_question'] = confirmatory_df['headline'].str.contains(r'\?', regex=True)
confirmatory_df['has_number'] = confirmatory_df['headline'].str.contains(r'\d', regex=True)
confirmatory_df['word_count'] = confirmatory_df['headline'].str.split().str.len()

# VADER gives a compound score from -1 (very negative) to +1 (very positive)
confirmatory_df['sentiment_compound'] = confirmatory_df['headline'].apply(
    lambda text: analyzer.polarity_scores(text)['compound']
)

# --- Build the two tables ---
conn = sqlite3.connect('upworthy.db')

# Re-save packages table WITH the new package_id (overwrites previous version)
packages_cols = ['package_id', 'clickability_test_id', 'headline', 'impressions',
                  'clicks', 'created_at', 'test_week', 'winner', 'significance']
confirmatory_df[packages_cols].to_sql('packages', conn, if_exists='replace', index=False)

# create headline_features with the NLP feature columns
features_cols = ['package_id', 'has_question', 'has_number', 'word_count', 'sentiment_compound']
confirmatory_df[features_cols].to_sql('headline_features', conn, if_exists='replace', index=False)

# --- Verify the join works ---
cursor = conn.cursor()
cursor.execute('''
    SELECT p.headline, p.impressions, p.clicks, f.has_question, f.sentiment_compound
    FROM packages p
    JOIN headline_features f ON p.package_id = f.package_id
    LIMIT 5
''')
for row in cursor.fetchall():
    print(row)

conn.close()

('Let’s See … Hire Cops, Pay Teachers, Buy Books For Schools. Or Kill People. Hard Choice, Right?', 3118, 8, 1, -0.7579)
('People Sent This Lesbian Questions And Her Raised Eyebrow Game Shames Us All', 4587, 130, 0, -0.4019)
('$3 Million Is What It Takes For A State To Legally Kill Someone', 3017, 19, 0, -0.6486)
('The Fact That Sometimes Innocent People Are Executed Is Enough To End The Death Penalty. But This?', 2974, 26, 1, -0.4118)
('Reason #351 To End The Death Penalty: It Costs $3 Million Per Case.', 3050, 10, 0, -0.7845)


#### Using the has_question and has_number columns will separate treatment and control while testing each feature

# Phase 2: SQL Aggregation

#### CTR By Feature

In [26]:
conn = sqlite3.connect('upworthy.db')
cursor = conn.cursor()

cursor.execute('''
    SELECT
        f.has_question,
        COUNT(*) AS n_packages,
        SUM(p.clicks) AS total_clicks,
        SUM(p.impressions) AS total_impressions,
        ROUND(1.0 * SUM(p.clicks) / SUM(p.impressions), 4) AS ctr
    FROM packages p
    JOIN headline_features f ON p.package_id = f.package_id
    GROUP BY f.has_question
''')
for row in cursor.fetchall():
    print(row)

(0, 89559, 4965959, 319130157, 0.0156)
(1, 15992, 775725, 57399001, 0.0135)


No question mark: higher sum, more clicks, more impressions, higher CTR rate. 1.56% CTR without a "?" vs 1.35% with a "?". Headlines without a question mark are more meaningful in terms of CTR (click-through rate)

#### CTR trend over time (test_week)

In [33]:
cursor.execute('''
    SELECT
        p.test_week,
        f.has_question,
        ROUND(1.0 * SUM(p.clicks) / SUM(p.impressions), 4) AS ctr
    FROM packages p
    JOIN headline_features f ON p.package_id = f.package_id
    GROUP BY p.test_week, f.has_question
    ORDER BY p.test_week
''')
results = cursor.fetchall()
results[:5]

[(201303, 0, 0.0231),
 (201303, 1, 0.0149),
 (201304, 0, 0.0251),
 (201304, 1, 0.0109),
 (201305, 0, 0.0236)]

#### Sentiment

In [34]:
cursor.execute('''
    SELECT
        CASE
            WHEN f.sentiment_compound >= 0.05 THEN 'positive'
            WHEN f.sentiment_compound <= -0.05 THEN 'negative'
            ELSE 'neutral'
        END AS sentiment_bucket,
        COUNT(*) AS n_packages,
        ROUND(1.0 * SUM(p.clicks) / SUM(p.impressions), 4) AS ctr
    FROM packages p
    JOIN headline_features f ON p.package_id = f.package_id
    GROUP BY sentiment_bucket
''')
results = cursor.fetchall()
results

[('negative', 35726, 0.016),
 ('neutral', 25208, 0.0149),
 ('positive', 44617, 0.0148)]

Sentiments: most are positive, then negative, fewest are neutral. Total CTR is positive for all 3. Negative headlines have the highest CTR (1.6%). Big find

#### Window Function

In [36]:
cursor.execute('''
    SELECT
        p.clickability_test_id,
        p.headline,
        p.clicks,
        p.impressions,
        ROUND(1.0 * p.clicks / p.impressions, 4) AS ctr,
        RANK() OVER (
            PARTITION BY p.clickability_test_id
            ORDER BY 1.0 * p.clicks / p.impressions DESC
        ) AS rank_within_test
    FROM packages p
    LIMIT 20
''')
results = cursor.fetchall()
results[:5]

[('5143605e220cb80002000076',
  "If You Know Anyone Who Is Afraid Of Gay People, Here's A Cartoon That Will Ease Them Back To Reality",
  120,
  4155,
  0.0289,
  1),
 ('5143605e220cb80002000076',
  "Here's The Science, Here's The Gay. Open Your Brain, They Were Born That Way!",
  54,
  4069,
  0.0133,
  2),
 ('5143605e220cb80002000076',
  "Hey Dude. If You Have An Older Brother, There's A Bigger Chance You're Gay.",
  41,
  4080,
  0.01,
  3),
 ('5143605e220cb80002000076',
  "I've Got Some News For You. Being Gay Is Genetic. Being Irrationally Afraid Of Gay, Not So Much.",
  40,
  4160,
  0.0096,
  4),
 ('5143605e220cb80002000076',
  'SCIENCE FACT: Gay Science, Like Straight Science, Is Really Just Plain Old Fact Science',
  32,
  4132,
  0.0077,
  5)]

Ranked by CTR (clicks / impressions): gay people headlines have highest CTR in this one test. Proves ranking works, no major findings.

#### Stratification

In [37]:
cursor.execute('''
    SELECT
        f.has_question,
        CASE
            WHEN f.sentiment_compound >= 0.05 THEN 'positive'
            WHEN f.sentiment_compound <= -0.05 THEN 'negative'
            ELSE 'neutral'
        END AS sentiment_bucket,
        COUNT(*) AS n_packages,
        ROUND(1.0 * SUM(p.clicks) / SUM(p.impressions), 4) AS ctr
    FROM packages p
    JOIN headline_features f ON p.package_id = f.package_id
    GROUP BY f.has_question, sentiment_bucket
''')
results = cursor.fetchall()
results

[(0, 'negative', 30525, 0.0164),
 (0, 'neutral', 20960, 0.0152),
 (0, 'positive', 38074, 0.0151),
 (1, 'negative', 5201, 0.0138),
 (1, 'neutral', 4248, 0.0134),
 (1, 'positive', 6543, 0.0134)]

In every single sentiment bucket, has_question=0 beats has_question=1. This means the "no question mark performs better" effect isn't just an artifact of question-headlines happening to also be more/less negative. It holds up consistently regardless of tone. 

# Phase 3: Statistical Testing

#### Will be getting p-values on both "has_question" and "has_number" to determine significance via z-test

In [39]:
from statsmodels.stats.proportion import proportions_ztest

#### has_question

In [40]:
# From your Query 1 results:
# has_question=0: clicks=4965959, impressions=319130157
# has_question=1: clicks=775725,  impressions=57399001

clicks = [4965959, 775725]
impressions = [319130157, 57399001]

z_stat, p_value = proportions_ztest(count=clicks, nobs=impressions)
print('z-statistic:', z_stat)
print('p-value:', p_value)

z-statistic: 116.472976585741
p-value: 0.0


Have to be careful with these results, z-test is large but that means the difference is not random, but not whether or not the difference is large enough to matter

#### Effect Size

In [41]:
# CTR for each group (from your Query 1 results)
ctr_no_question = 4965959 / 319130157
ctr_question = 775725 / 57399001

print('CTR no question:', round(ctr_no_question, 5))
print('CTR question:', round(ctr_question, 5))
print('Absolute difference (percentage points):', round((ctr_no_question - ctr_question) * 100, 3))
print('Relative lift:', round((ctr_no_question - ctr_question) / ctr_question * 100, 2), '%')

CTR no question: 0.01556
CTR question: 0.01351
Absolute difference (percentage points): 0.205
Relative lift: 15.14 %


Relative lift: headlines without a question mark got 15.14% more clicks proportionally than ones with a question mark

#### Confidence Interval

In [42]:
from statsmodels.stats.proportion import confint_proportions_2indep

ci_low, ci_high = confint_proportions_2indep(
    count1=4965959, nobs1=319130157,   # has_question = 0
    count2=775725,  nobs2=57399001,    # has_question = 1
    method='wald'
)

print('95% CI for the difference in CTR:', ci_low, ci_high)

95% CI for the difference in CTR: 0.002013500188113673 0.002079124731940859


Interval is narrow (about 0.0007 percentage points wide). Excludes zero. Small interval due to enormous sample size

#### has_number

In [43]:
cursor.execute('''
    SELECT
        f.has_number,
        COUNT(*) AS n_packages,
        SUM(p.clicks) AS total_clicks,
        SUM(p.impressions) AS total_impressions,
        ROUND(1.0 * SUM(p.clicks) / SUM(p.impressions), 4) AS ctr
    FROM packages p
    JOIN headline_features f ON p.package_id = f.package_id
    GROUP BY f.has_number
''')
results = cursor.fetchall()
results

[(0, 86269, 4686932, 307927858, 0.0152), (1, 19282, 1054752, 68601300, 0.0154)]

In [44]:
clicks = [4686932, 1054752]
impressions = [307927858, 68601300]

z_stat, p_value = proportions_ztest(count=clicks, nobs=impressions)
print('z-statistic:', z_stat)
print('p-value:', p_value)

z-statistic: -9.426786510364918
p-value: 4.2284502580909284e-21


Group 0 has a lower CTR, so z is negative. p-value is 4.22 * 10^-21 so that is significant

In [45]:
ctr_no_number = 4686932 / 307927858
ctr_number = 1054752 / 68601300

print('CTR no number:', round(ctr_no_number, 5))
print('CTR number:', round(ctr_number, 5))
print('Absolute difference (percentage points):', round((ctr_number - ctr_no_number) * 100, 3))
print('Relative lift:', round((ctr_number - ctr_no_number) / ctr_no_number * 100, 2), '%')

CTR no number: 0.01522
CTR number: 0.01538
Absolute difference (percentage points): 0.015
Relative lift: 1.01 %


has_questions effect is 15x larger than has_number (due to the lift). Important because p-values would have stated both features matter, but the relative lift (effect size) demonstrates that question marks account for way more than numbers in the text. 

#### Sentiment Chi-Square Test

In [46]:
cursor.execute('''
    SELECT
        CASE
            WHEN f.sentiment_compound >= 0.05 THEN 'positive'
            WHEN f.sentiment_compound <= -0.05 THEN 'negative'
            ELSE 'neutral'
        END AS sentiment_bucket,
        SUM(p.clicks) AS total_clicks,
        SUM(p.impressions) AS total_impressions
    FROM packages p
    JOIN headline_features f ON p.package_id = f.package_id
    GROUP BY sentiment_bucket
''')
results = cursor.fetchall()
results

[('negative', 2042785, 127435547),
 ('neutral', 1337364, 89996120),
 ('positive', 2361535, 159097491)]

In [47]:
from scipy.stats import chi2_contingency

# Build a contingency table: rows = sentiment groups, columns = [clicks, non-clicks]
data = {
    'negative': (2042785, 127435547),
    'neutral':  (1337364, 89996120),
    'positive': (2361535, 159097491),
}

contingency_table = []
for sentiment, (clicks, impressions) in data.items():
    non_clicks = impressions - clicks
    contingency_table.append([clicks, non_clicks])

print(contingency_table)

chi2_stat, p_value, dof, expected = chi2_contingency(contingency_table)
print('chi2 statistic:', chi2_stat)
print('p-value:', p_value)
print('degrees of freedom:', dof)

[[2042785, 125392762], [1337364, 88658756], [2361535, 156735956]]
chi2 statistic: 7825.099265034444
p-value: 0.0
degrees of freedom: 2


p-value is not really 0, it's close to 0. This shows that sentiment is significantly associated with CTR overall, degrees of freedom = 2 confirms it correctly treated it as a 3-group comparison. 

#### pairwise z-test

In [48]:
clicks = [2042785, 2361535]
impressions = [127435547, 159097491]

z_stat, p_value = proportions_ztest(count=clicks, nobs=impressions)
print('z-statistic:', z_stat)
print('p-value:', p_value)

z-statistic: 81.1363007913274
p-value: 0.0


In [49]:
ctr_negative = 2042785 / 127435547
ctr_positive = 2361535 / 159097491

print('CTR negative:', round(ctr_negative, 5))
print('CTR positive:', round(ctr_positive, 5))
print('Absolute difference (percentage points):', round((ctr_negative - ctr_positive) * 100, 3))
print('Relative lift:', round((ctr_negative - ctr_positive) / ctr_positive * 100, 2), '%')

CTR negative: 0.01603
CTR positive: 0.01484
Absolute difference (percentage points): 0.119
Relative lift: 7.99 %


In [50]:
ci_low, ci_high = confint_proportions_2indep(
    count1=2042785, nobs1=127435547,   # negative
    count2=2361535, nobs2=159097491,   # positive
    method='wald'
)
print('95% CI for the difference in CTR:', ci_low, ci_high)

95% CI for the difference in CTR: 0.001157841885952889 0.0012154107287583888


For sentiment, lift is less than question mark and more than number. Question marks hurt CTR the most. Negative sentiment helps meaningfully (~8% relative lift over positive ones).

# Phase 4

#### No ODBC Connection, so manually converting queries into CSVs

#### 1st Query: has_question CTR trend over time

In [62]:
df_trend = pd.read_sql('''
    SELECT
        DATE(p.created_at, 'weekday 0', '-6 days') AS test_week_start,
        f.has_question,
        SUM(p.clicks) AS total_clicks,
        SUM(p.impressions) AS total_impressions,
        ROUND(1.0 * SUM(p.clicks) / SUM(p.impressions), 4) AS ctr
    FROM packages p
    JOIN headline_features f ON p.package_id = f.package_id
    GROUP BY test_week_start, f.has_question
    ORDER BY test_week_start
''', conn)

df_trend.to_csv('has_question_ctr_by_week_2.csv', index=False)

#### Query **2**: overall feature comparison summary - one table for a clean summary chart

In [52]:
df_question = pd.read_sql('''
    SELECT 'has_question' AS feature, f.has_question AS value,
           SUM(p.clicks) AS clicks, SUM(p.impressions) AS impressions,
           ROUND(1.0*SUM(p.clicks)/SUM(p.impressions),4) AS ctr
    FROM packages p JOIN headline_features f ON p.package_id=f.package_id
    GROUP BY f.has_question
''', conn)

df_number = pd.read_sql('''
    SELECT 'has_number' AS feature, f.has_number AS value,
           SUM(p.clicks) AS clicks, SUM(p.impressions) AS impressions,
           ROUND(1.0*SUM(p.clicks)/SUM(p.impressions),4) AS ctr
    FROM packages p JOIN headline_features f ON p.package_id=f.package_id
    GROUP BY f.has_number
''', conn)

df_sentiment = pd.read_sql('''
    SELECT 'sentiment' AS feature,
           CASE WHEN f.sentiment_compound >= 0.05 THEN 'positive'
                WHEN f.sentiment_compound <= -0.05 THEN 'negative'
                ELSE 'neutral' END AS value,
           SUM(p.clicks) AS clicks, SUM(p.impressions) AS impressions,
           ROUND(1.0*SUM(p.clicks)/SUM(p.impressions),4) AS ctr
    FROM packages p JOIN headline_features f ON p.package_id=f.package_id
    GROUP BY value
''', conn)

df_summary = pd.concat([df_question, df_number, df_sentiment], ignore_index=True)
df_summary.to_csv('feature_ctr_summary.csv', index=False)
df_summary

,feature,value,clicks,impressions,ctr
0,has_question,0,4965959,319130157,0.0156
1,has_question,1,775725,57399001,0.0135
2,has_number,0,4686932,307927858,0.0152
3,has_number,1,1054752,68601300,0.0154
4,sentiment,negative,2042785,127435547,0.0160
5,sentiment,neutral,1337364,89996120,0.0149
6,sentiment,positive,2361535,159097491,0.0148


#### Query **3**: Stratified View (has_question x sentiment)

In [54]:
df_strat = pd.read_sql('''
    SELECT
        f.has_question,
        CASE WHEN f.sentiment_compound >= 0.05 THEN 'positive'
             WHEN f.sentiment_compound <= -0.05 THEN 'negative'
             ELSE 'neutral' END AS sentiment_bucket,
        SUM(p.clicks) AS clicks, SUM(p.impressions) AS impressions,
        ROUND(1.0*SUM(p.clicks)/SUM(p.impressions),4) AS ctr
    FROM packages p JOIN headline_features f ON p.package_id=f.package_id
    GROUP BY f.has_question, sentiment_bucket
''', conn)

df_strat.to_csv('has_question_by_sentiment.csv', index=False)

df_strat

,has_question,sentiment_bucket,clicks,impressions,ctr
0,0,negative,1788087,108945034,0.0164
1,0,neutral,1134992,74849530,0.0152
2,0,positive,2042880,135335593,0.0151
3,1,negative,254698,18490513,0.0138
4,1,neutral,202372,15146590,0.0134
5,1,positive,318655,23761898,0.0134
